In [1]:
import pandas as pd 
import numpy as np
import dagshub
import plotly.express as px 
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
# from sentence_transformers import SentenceTransformer
from scipy.sparse import hstack, csr_matrix
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("cleaned_data3.csv")
df["Sentiment"] = df["Sentiment"].astype("int8")

In [3]:
import warnings
warnings.filterwarnings("ignore")
df["Sentiment"].unique()

array([ 0,  1, -1], dtype=int8)

In [4]:
# there was only one NAn
df.dropna(inplace= True)

In [5]:
comments = df["CommentText"].tolist()

model = SentenceTransformer("all-MiniLM-L6-v2")
comments_embeddings = model.encode(
    comments,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
).astype(np.float32)

# saving the embeddigns on disk
np.save("comments_embeddings.npy", comments_embeddings)

NameError: name 'SentenceTransformer' is not defined

In [10]:
loaded_embeddings = np.load("comments_embeddings1.npy")

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 971460 entries, 0 to 971512
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   CommentText  971460 non-null  object
 1   Sentiment    971460 non-null  int8  
dtypes: int8(1), object(1)
memory usage: 15.7+ MB


In [7]:
dagshub.init(repo_owner= 'vaibhav-patel01' , repo_name= 'YT_Comments_Analysis_chrome_extension' , mlflow= True)

mlflow.set_tracking_uri("https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow")
mlflow.set_experiment("finding_best_feature_engineering_algo")

Accessing as vaibhav-patel01

Initialized MLflow to track repo "vaibhav-patel01/YT_Comments_Analysis_chrome_extension"

Repository vaibhav-patel01/YT_Comments_Analysis_chrome_extension initialized!

<Experiment: artifact_location='mlflow-artifacts:/cf5821c3c16a404c81ff005e5f1f60f0', creation_time=1783399519152, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783399519152, lifecycle_stage='active', name='finding_best_feature_engineering_algo', tags={}, trace_location=None, workspace='default'>

In [11]:
x= loaded_embeddings 
y = df["Sentiment"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tags({
            "mlflow.runName" : "vector_embeddings_RandomForest",
            "experiment_type" : "feature_engineering",
            "model_type" : "RandomForestClassifier"

            })
        n_estimators = 200
        max_depth = 15

        # Log vectorizer parameters
        mlflow.log_params({
            "vectorizer_type" : "sentence_transformer",
            "n_estimators" : n_estimators,
            "max_depth" : max_depth
            })

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(x_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        fig = px.imshow(
            conf_matrix, 
            text_auto=True, # Equivalent to annot=True
            color_continuous_scale="Blues", # Equivalent to cmap="Blues"
            labels=dict(x="Predicted", y="Actual"),
            title= "Confusion Matrix:",
            width=800, # Approximate equivalent to figsize=(8, 6)
            height=600
        )
        mlflow.log_figure(fig, "basline_confusion_matrix.html")

🏃 View run vector_embeddings_RandomForest at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1/runs/ae717505a68449b49db078829a27036a
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1


KeyboardInterrupt: 

In [31]:
# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X = vectorizer.fit_transform(df['CommentText'])
    y = df['Sentiment']

    # Step 3: Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tags({
            "mlflow.runName" : f"{vectorizer_name}_{ngram_range}_RandomForest",
            "experiment_type" : "feature_engineering",
            "model_type" : "RandomForestClassifier",
            "description" : f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}"

            })
        n_estimators = 200
        max_depth = 15

        # Log vectorizer parameters
        mlflow.log_params({
            "vectorizer_type" : vectorizer_type,
            "ngram_range" : ngram_range, 
            "vectorizer_max_features": vectorizer_max_features,
            "n_estimators" : n_estimators,
            "max_depth" : max_depth
            })

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        fig = px.imshow(
            conf_matrix, 
            text_auto=True, # Equivalent to annot=True
            color_continuous_scale="Blues", # Equivalent to cmap="Blues"
            labels=dict(x="Predicted", y="Actual"),
            title= "Confusion Matrix:",
            width=800, # Approximate equivalent to figsize=(8, 6)
            height=600
        )
        mlflow.log_figure(fig, "basline_confusion_matrix.html")

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)] # unigrams, bigrams, trigrams
max_features = 50000 # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")

2026/07/07 10:19:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 1)_RandomForest at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1/runs/29d8aaa228e142628771003a90b63e66
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1


2026/07/07 10:27:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 1)_RandomForest at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1/runs/883320b3d03f4cd98a862b8102c5f153
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1


2026/07/07 10:34:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1/runs/b247215be5164613845b886a13864529
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1


2026/07/07 10:51:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 3)_RandomForest at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1/runs/35d9dfdd1be747f0acd8c30ab69db546
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1


2026/07/07 10:59:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 3)_RandomForest at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1/runs/c25f932703a945bbba5f3dcce3839016
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/1


In [ ]:
fig = px.imshow(
            conf_matrix, 
            text_auto=True, # Equivalent to annot=True
            color_continuous_scale="Blues", # Equivalent to cmap="Blues"
            labels=dict(x="Predicted", y="Actual"),
            title= "Confusion Matrix:",
            width=800, # Approximate equivalent to figsize=(8, 6)
            height=600
        )
        mlflow.log_figure(fig, "basline_confusion_matrix.html")

In [ ]:
# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X = vectorizer.fit_transform(df['clean_comment'])
    y = df['category']

    # Step 3: Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)] # unigrams, bigrams, trigrams
max_features = 5000 # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")